# PPLSTM vs sLSTM — **single-block parity learning**

**Kaggle GPU notebook, no mLSTM layers and no language-model training.** One official xLSTM **vanilla sLSTM block** versus the *same official one-block residual/FFN wrapper* with its sLSTM layer replaced by the original PPLSTM cell. Each receives a small identical token embedding and final-token classifier; those are necessary to express the supervised task.

Both models train on independently generated **binary parity**, using exactly matched training and test examples, seed schedule, optimizer, context width and steps. Training starts with short *active-bit* parity and increases difficulty to 128 independent bits. The GPU sequence remains 128 tokens during training; zero bits pad shorter active-bit phases on the left. A direct 128-bit-only training mode is available as a separate optional study.

**What counts as success?** Consistently high *held-out* accuracy well above 50% at 128 bits, replicated over seeds. Longer context results show length generalization. A failed parity run does **not** prove an architecture incapable of parity.

- sLSTM: official `xlstm==2.0.6` block, `backend='vanilla'` (not official optimized CUDA kernel). An algebraically equivalent GPU-safe branch is applied and verified so CUDA Graph capture can work on T4.
- PPLSTM: your original three-channel permutation cell with associative scans, inside the same one-block wrapper.
- CUDA Graphs capture **forward + backward only**; AdamW and gradient clipping run outside capture. If capture fails, training uses labelled eager mode. No performance claims without timings.
- This is **one block per architecture** — not your four-block mLSTM/PPLSTM model. Width is matched; **parameter counts are reported, not matched**.

**License note:** PPLSTM source is AGPL-3.0-only; see your original project terms. Official sLSTM is loaded from the installed xLSTM package, not copied here. Experiment support code is not a change to either project's licensing.

## 1. Kaggle settings and imports

In [ ]:
# Configuration: the default is intended to probe whether a single block can learn parity.
WIDTH = 72                    # multiple of 3 (PPLSTM) and 4 (official sLSTM heads)
HEADS = 4
TRAIN_CONTEXT = 128           # includes final query token: 127 data bits at most
BATCH = 32
TRAIN_STEPS = 8_000           # PER architecture × seed
SEEDS = (13, 29)             # try 3+ seeds later if promising
LR = 1e-3
WEIGHT_DECAY = 0.01
REPORT_EVERY = 500
SAVE_EVERY = 1_000
EVAL_LENGTHS = (32, 64, 128, 256, 512)
FINAL_EVAL_BATCHES = 32      # 32 × batch 32 = 1,024 held-out examples per context
INTERIM_EVAL_BATCHES = 8    # 256 examples, repeated consistently across models
USE_CUDA_GRAPHS = True
RUN_DIRECT_BASELINES = False # additionally compare direct 128-bit training from step 1 (more GPU use)
SMOKE_ONLY = False           # True: verify one optimizer step per model and stop
RESUME_DIR = ''               # optional folder attached from saved Kaggle output

# Task: bits 0 and 1, final query token 2. No answer token or privileged information in input.
# Curriculum works within 127 bit positions, query at the final 128th position.
def active_bits_for_step(step, steps=TRAIN_STEPS):
    if step <= steps * 0.125: return 8
    if step <= steps * 0.25: return 16
    if step <= steps * 0.375: return 32
    if step <= steps * 0.50: return 64
    return TRAIN_CONTEXT - 1

import os, sys, subprocess, importlib.util, math, gc, json, time, csv, zipfile, random, hashlib, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

if importlib.util.find_spec('xlstm') is None:
    print('Official xlstm package missing; installing from PyPI. Kaggle Internet must be ON.')
    try:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','xlstm==2.0.6'])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Cannot install xlstm. In Kaggle turn Internet ON, or add an xlstm 2.0.6 wheel as Input. This notebook will not silently substitute an sLSTM.') from exc

import xlstm
assert torch.cuda.is_available(), 'Select Kaggle Settings → Accelerator → GPU, then rerun from the top.'
assert WIDTH % 12 == 0, 'Width must divide into 4 heads and groups of 3.'
assert TRAIN_CONTEXT >= 16 and BATCH % 2 == 0
assert REPORT_EVERY > 0 and SAVE_EVERY > 0
DEVICE=torch.device('cuda:0')
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
OUT = Path('/kaggle/working/single_block_parity' if Path('/kaggle/working').exists() else '/mnt/data/single_block_parity')
OUT.mkdir(parents=True, exist_ok=True)
resume_root = Path(RESUME_DIR) if RESUME_DIR else OUT
print('GPU:',torch.cuda.get_device_name(0),'capability:',torch.cuda.get_device_capability(0))
print('PyTorch:',torch.__version__,'xLSTM installed at:',xlstm.__file__,'output:',OUT)
print('Runs:',len(SEEDS)*2*(2 if RUN_DIRECT_BASELINES else 1),'steps:',TRAIN_STEPS,'per run')

def count_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
def save_csv(path,rows):
    if rows: pd.DataFrame(rows).to_csv(path,index=False)
def atomic_save(obj,path):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    torch.save(obj,tmp); os.replace(tmp,path)
def save_json(path,obj):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str)); os.replace(tmp,path)

def export_results():
    zpath=OUT/'PPLSTM_sLSTM_Single_Block_Parity_Results.zip'
    tmp=OUT/(zpath.name+'.tmp')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(list(OUT.glob('*.csv'))+list(OUT.glob('*.json'))):
            z.write(p,arcname=p.name)
    os.replace(tmp,zpath)
    return zpath

## 2. Your PPLSTM cell (unaltered equations)

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Adapted from the original PPLSTM project by Rickey (https://github.com/rickey1990/PPLSTM/blob/main/pplstm.py)
# The transition equations and scan implementation are kept as published.
"""PPLSTM: associative 3-channel permutation-mixed recurrent cell."""
import itertools
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F


def shift_right(x: Tensor) -> Tensor:
    return torch.cat((torch.zeros_like(x[:, :1]), x[:, :-1]), dim=1)


def scalar_affine_scan(a: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    A, B = a, b
    step = 1
    while step < a.shape[1]:
        A, B = (
            torch.cat((A[:, :step], A[:, step:] * A[:, :-step]), dim=1),
            torch.cat((B[:, :step], B[:, step:] + A[:, step:] * B[:, :-step]), dim=1),
        )
        step *= 2
    return A, B


def matrix_affine_scan(A: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    step = 1
    while step < A.shape[1]:
        next_A = torch.cat((A[:, :step], A[:, step:] @ A[:, :-step]), dim=1)
        next_b = torch.cat((
            b[:, :step],
            b[:, step:] + (A[:, step:] @ b[:, :-step].unsqueeze(-1)).squeeze(-1),
        ), dim=1)
        A, b = next_A, next_b
        step *= 2
    return A, b


class PPLSTM(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        if width < 3 or width % 3 != 0:
            raise ValueError('width must be a positive multiple of 3')
        self.width = width
        self.groups = width // 3
        self.xproj = nn.Linear(width, 4 * width)
        self.mix_logit = nn.Parameter(torch.full((4, width), math.log(3.0)))
        self.sign_proj = nn.Linear(width, width)
        self.initial_cell = nn.Parameter(torch.linspace(-1.5, 1.5, 3).repeat(self.groups))
        self.input_bias = nn.Parameter(torch.full((width,), -4.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 4.0))
        nn.init.zeros_(self.sign_proj.weight)
        nn.init.constant_(self.sign_proj.bias, 2.8)

        bank = []
        for order in itertools.permutations(range(3)):
            matrix = torch.zeros(3, 3)
            for row, column in enumerate(order):
                matrix[row, column] = 1.0
            bank.append(matrix)
        self.register_buffer('permutation_bank', torch.stack(bank))

        self.selector = nn.Linear(width, self.groups * 6)
        nn.init.zeros_(self.selector.weight)
        nn.init.constant_(self.selector.bias, -1.0)
        with torch.no_grad():
            for group in range(self.groups):
                self.selector.bias[group * 6] = 3.0

    def _token_maps(self, x: Tensor):
        if x.ndim != 3 or x.shape[-1] != self.width:
            raise ValueError('Expected x with shape [batch, time, width]')
        batch, length, width = x.shape
        group_count = self.groups
        previous = shift_right(x)
        coeff = self.mix_logit.sigmoid().view(1, 1, 4, width)
        gates = (self.xproj(x).reshape(batch, length, 4, width) * coeff
                 + self.xproj(previous).reshape(batch, length, 4, width) * (1 - coeff))
        input_logits, forget_logits, candidate_logits, output_logits = gates.unbind(2)
        mixed = x * self.mix_logit[0].sigmoid() + previous * (1 - self.mix_logit[0].sigmoid())
        signs = self.sign_proj(mixed).tanh().reshape(batch, length, group_count, 3)
        z = candidate_logits.tanh().reshape(batch, length, group_count, 3)
        log_i = (input_logits + self.input_bias).reshape(batch, length, group_count, 3).mean(-1)
        log_f = F.logsigmoid((forget_logits + self.forget_bias)
                           .reshape(batch, length, group_count, 3).mean(-1))
        f_prefix = log_f.cumsum(dim=1)
        m = f_prefix + torch.cummax(torch.cat((
            torch.zeros_like(f_prefix[:, :1]), log_i - f_prefix,
        ), dim=1), dim=1).values[:, 1:]
        f = (log_f + shift_right(m) - m).exp()
        i = (log_i - m).exp()
        weights = self.selector(mixed).reshape(batch, length, group_count, 6).softmax(-1)
        P = torch.einsum('btgp,pij->btgij', weights, self.permutation_bank)
        A = f[..., None, None] * P * signs[..., None, :]
        b = i[..., None] * z
        return A, b, f, i, output_logits

    def forward(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        prefix_A, prefix_b = matrix_affine_scan(A, b)
        c0 = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        c = (prefix_A @ c0[:, None, :, :, None]).squeeze(-1) + prefix_b
        norm_A, norm_b = scalar_affine_scan(f, i)
        n = norm_A + norm_b
        ratio = (c / n[..., None].clamp_min(1e-12)).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

    def serial_reference(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        c = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        n = x.new_ones(batch, self.groups)
        states = []
        for t in range(length):
            c = (A[:, t] @ c.unsqueeze(-1)).squeeze(-1) + b[:, t]
            n = f[:, t] * n + i[:, t]
            states.append(c / n[..., None].clamp_min(1e-12))
        ratio = torch.stack(states, dim=1).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

## 3. Official vanilla sLSTM: verify a CUDA Graph-safe branch

In [ ]:
# Upstream pointwise function: https://github.com/NX-AI/xlstm/blob/main/xlstm/blocks/slstm/src/vanilla/slstm.py
# The official implementation uses `if torch.all(n == 0)` which synchronizes the GPU during capture.
# The following computes EXACTLY the same global condition on GPU with torch.where.
# All sLSTM training runs use this mathematically equivalent version (including eager fallback).

def graph_safe_slstm_pointwise(Wx, Ry, b, states, constants):
    raw=Wx+Ry+b
    y,c,n,m=torch.unbind(states.view(4,states.shape[1],-1),dim=0)
    iraw,fraw,zraw,oraw=torch.unbind(raw.view(raw.shape[0],4,-1),dim=1)
    logfplusm=m+F.logsigmoid(fraw)
    mnew=torch.where(torch.all(n == 0.0), iraw, torch.max(iraw,logfplusm))
    ogate=torch.sigmoid(oraw)
    igate=torch.minimum(torch.exp(iraw-mnew),torch.ones_like(iraw))
    fgate=torch.minimum(torch.exp(logfplusm-mnew),torch.ones_like(iraw))
    cnew=fgate*c+igate*torch.tanh(zraw)
    nnew=fgate*n+igate
    ynew=ogate*cnew/nnew
    return (torch.stack((ynew,cnew,nnew,mnew),dim=0),
            torch.stack((igate,fgate,zraw,ogate),dim=0))

from xlstm.blocks.slstm.src.vanilla.slstm import slstm_forward_pointwise
from xlstm.blocks.slstm.cell import sLSTMCell_vanilla
PATCH_CHECKS=[]
for mode in ('all_zero_n','normal_n','mixed_n'):
    gen=torch.Generator(device='cpu').manual_seed(1020+len(PATCH_CHECKS))
    src=[torch.randn(3,4*12,generator=gen,dtype=torch.double)*0.15,
         torch.randn(3,4*12,generator=gen,dtype=torch.double)*0.15,
         torch.randn(1,4*12,generator=gen,dtype=torch.double)*0.15,
         torch.randn(4,3,12,generator=gen,dtype=torch.double)*0.15]
    src[3][2].fill_(0.0 if mode=='all_zero_n' else 1.0)
    if mode=='mixed_n': src[3][2,0,0]=0
    def measure(fn):
        args=[a.clone().detach().requires_grad_() for a in src]
        out=fn(*args,constants={})
        score=out[0].square().sum()+out[1].square().sum()*0.17
        grads=torch.autograd.grad(score,args)
        return out,grads
    a,ga=measure(slstm_forward_pointwise)
    b,gb=measure(graph_safe_slstm_pointwise)
    diff=max((x-y).abs().max().item() for x,y in zip(a,b))
    grad=max((x-y).abs().max().item() for x,y in zip(ga,gb))
    passed=diff<1e-10 and grad<1e-10
    PATCH_CHECKS.append(dict(test=mode,passed=passed,max_output_abs=diff,max_grad_abs=grad))
    assert passed, 'sLSTM pointwise patch is NOT equivalent; benchmark blocked.'
print('Official sLSTM pointwise equivalence:',PATCH_CHECKS)

def patch_slstm(model):
    found=0
    for mod in model.modules():
        if isinstance(mod,sLSTMCell_vanilla):
            mod.pointwise=graph_safe_slstm_pointwise
            found+=1
    assert found==1, f'Expected exactly one official vanilla sLSTM cell; got {found}'
    return found

## 4. One-block models and matched parity classifier

In [ ]:
from xlstm import xLSTMBlockStack, xLSTMBlockStackConfig, sLSTMBlockConfig, sLSTMLayerConfig, FeedForwardConfig

class PPLSTMReplacementLayer(nn.Module):
    """Original research PPLSTM in the one-block sLSTM layer slot.
    Keep the official sLSTM front-end convolution, group norm and dropout.
    This matches the previously tested mLSTM+PPLSTM replacement strategy.
    """
    def __init__(self, original_layer):
        super().__init__()
        self.config=original_layer.config
        self.conv1d=original_layer.conv1d if self.config.conv1d_kernel_size>0 else None
        self.conv_act_fn=original_layer.conv_act_fn if self.conv1d is not None else None
        self.group_norm=original_layer.group_norm
        self.dropout=original_layer.dropout
        self.pplstm=PPLSTM(self.config.embedding_dim)
    def forward(self,x,**kwargs):
        B,T,W=x.shape
        hx=self.conv_act_fn(self.conv1d(x)) if self.conv1d is not None else x
        y=self.dropout(self.pplstm(hx))
        H=self.config.num_heads
        y=y.reshape(B,T,H,W//H).transpose(1,2)
        return self.group_norm(y).transpose(1,2).reshape(B,T,W)

class SingleBlockParity(nn.Module):
    def __init__(self,cell_type):
        super().__init__()
        self.cell_type=cell_type
        self.embed=nn.Embedding(3,WIDTH)
        cfg=xLSTMBlockStackConfig(
            mlstm_block=None,
            slstm_block=sLSTMBlockConfig(
                slstm=sLSTMLayerConfig(backend='vanilla',num_heads=HEADS,
                    conv1d_kernel_size=4,bias_init='powerlaw_blockdependent'),
                feedforward=FeedForwardConfig(proj_factor=1.3,act_fn='gelu')),
            embedding_dim=WIDTH,context_length=max(EVAL_LENGTHS+(TRAIN_CONTEXT,)),
            num_blocks=1,slstm_at=[0],dropout=0.0)
        self.backbone=xLSTMBlockStack(cfg)
        assert len(self.backbone.blocks)==1
        if cell_type=='slstm':
            patch_slstm(self.backbone)
        elif cell_type=='pplstm':
            original=self.backbone.blocks[0].xlstm
            self.backbone.blocks[0].xlstm=PPLSTMReplacementLayer(original)
        else: raise ValueError(cell_type)
        self.norm=nn.LayerNorm(WIDTH)
        self.head=nn.Linear(WIDTH,2)
    def forward(self,x):
        h=self.backbone(self.embed(x))
        return self.head(self.norm(h[:,-1]))

def make_seeded_model(arch,seed):
    # Architectures consume different random numbers while constructing their cells.
    # Force identical initial embedding and classifier weights for matched-seed comparisons.
    torch.manual_seed(seed)
    model=SingleBlockParity(arch).to(DEVICE)
    with torch.random.fork_rng(devices=[DEVICE.index or 0]):
        torch.manual_seed(seed+100_000)
        nn.init.normal_(model.embed.weight,std=0.02)
        nn.init.xavier_uniform_(model.head.weight)
        nn.init.zeros_(model.head.bias)
    return model

# Verify there is exactly one recurrent block in each model, not a full stack.
for arch in ('slstm','pplstm'):
    m=make_seeded_model(arch,101)
    x=torch.randint(0,2,(2,16),device=DEVICE)
    x[:,-1]=2
    result=m(x)
    assert result.shape==(2,2),result.shape
    assert torch.isfinite(result).all()
    print(arch,'parameters:',count_params(m),'block:',type(m.backbone.blocks[0].xlstm).__name__)
    del m
    torch.cuda.empty_cache()

## 5. Parity data — class balance, no answer leakage, independent evaluation

In [ ]:
# Generate exactly half 0 and half 1 parity labels in every even-sized batch.
# This prevents a class-imbalance shortcut and makes the 50% chance baseline exact.
# Training examples vary by run and step; eval examples are fixed, identical for both models.
# The final query token does not contain the answer.

def parity_batch(batch, length, seed, active_bits=None, device=DEVICE):
    rng=np.random.default_rng(int(seed))
    assert batch%2==0 and length>=3
    nbits=length-1
    k=nbits if active_bits is None else min(nbits,int(active_bits))
    bits=np.zeros((batch,nbits),dtype=np.int64)
    # Right-align active bits: the first curriculum stage needs only 8-bit parity,
    # not 8-bit parity followed by 119 extra memory steps.
    if k>0: bits[:,nbits-k:]=rng.integers(0,2,size=(batch,k),dtype=np.int64)
    labels=np.array([0,1]*(batch//2),dtype=np.int64)
    rng.shuffle(labels)
    current=bits.sum(axis=1)%2
    flips=(current!=labels)
    flip_locations=rng.integers(nbits-k,nbits,size=(batch,))
    bits[np.arange(batch),flip_locations] ^= flips.astype(np.int64)
    assert np.array_equal(bits.sum(axis=1)%2,labels)
    x=np.concatenate([bits,np.full((batch,1),2,dtype=np.int64)],axis=1)
    return torch.from_numpy(x).to(device),torch.from_numpy(labels).to(device)

# Fixed held-out values: negative seed ranges distinct from training.
@torch.no_grad()
def evaluate(model,length,batches=INTERIM_EVAL_BATCHES,active_bits=None):
    was_training=model.training
    model.eval()
    totals=np.zeros((2,2),dtype=np.int64) # [truth, prediction]
    loss_sum=0.; n=0
    for k in range(batches):
        x,y=parity_batch(BATCH,length,seed=900_000_000+length*7001+k,active_bits=active_bits)
        logits=model(x)
        pred=logits.argmax(dim=-1)
        for truth in (0,1):
            for guess in (0,1):
                totals[truth,guess] += int(((y==truth)&(pred==guess)).sum().item())
        loss_sum+=float(F.cross_entropy(logits,y,reduction='sum').item())
        n+=BATCH
    model.train(was_training)
    return dict(accuracy_pct=100*(totals[0,0]+totals[1,1])/n,
                eval_loss=loss_sum/n,examples=n,
                pred_zero_pct=100*totals[:,0].sum()/n,
                pred_one_pct=100*totals[:,1].sum()/n,
                true_zero_recall_pct=100*totals[0,0]/max(1,totals[0].sum()),
                true_one_recall_pct=100*totals[1,1]/max(1,totals[1].sum()))

x,y=parity_batch(32,128,123)
assert x[:,-1].eq(2).all() and x[:,:-1].le(1).all()
assert y.sum().item()==16 and torch.equal((x[:,:-1].sum(1)%2),y)
print('Dataset sanity PASS:',x.shape,'class totals',torch.bincount(y).tolist())

## 6. Numerical correctness of your PPLSTM scan

In [ ]:
SCAN_CHECKS=[]
for length in (16,65):
    torch.manual_seed(201+length)
    c=PPLSTM(WIDTH).to(DEVICE).double()
    x=torch.randn(2,length,WIDTH,device=DEVICE,dtype=torch.float64,requires_grad=True)*0.15
    parallel=c(x)
    serial=c.serial_reference(x)
    max_out=float((parallel-serial).abs().max().item())
    pg=torch.autograd.grad(parallel.square().sum(),tuple(c.parameters()),retain_graph=True)
    sg=torch.autograd.grad(serial.square().sum(),tuple(c.parameters()))
    max_grad=max(float((p-s).abs().max().item()) for p,s in zip(pg,sg))
    passed=max_out<1e-8 and max_grad<1e-7
    SCAN_CHECKS.append(dict(length=length,max_output_abs=max_out,max_gradient_abs=max_grad,passed=passed))
    assert passed,SCAN_CHECKS[-1]
    del c
print('PPLSTM parallel-vs-serial output and gradient:',SCAN_CHECKS)

## 7. CUDA Graph forward/backward capture (AdamW remains outside)

In [ ]:
# Same capture approach as your prior validated PPLSTM Kaggle notebook.
# The sLSTM graph path uses the verified tensor-only pointwise branch above.
# Both architectures may separately fall back to eager, and the output records the engine.
class ForwardBackwardGraph:
    def __init__(self,model,x,y):
        assert x.is_cuda
        self.model=model
        self.static_x=torch.empty_like(x)
        self.static_y=torch.empty_like(y)
        self.graph=torch.cuda.CUDAGraph()
        side=torch.cuda.Stream(device=x.device)
        side.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(side):
            for _ in range(3):
                model.zero_grad(set_to_none=True)
                loss=F.cross_entropy(model(self.static_x.fill_(0)),self.static_y.fill_(0))
                loss.backward()
        torch.cuda.current_stream().wait_stream(side)
        model.zero_grad(set_to_none=True)
        with torch.cuda.graph(self.graph,stream=side):
            self.loss=F.cross_entropy(model(self.static_x),self.static_y)
            self.loss.backward()
        torch.cuda.synchronize()
    def replay(self,x,y):
        self.static_x.copy_(x)
        self.static_y.copy_(y)
        self.graph.replay()
        return self.loss

def optimizer_step(model,opt,x,y,graph=None):
    if graph is None:
        opt.zero_grad(set_to_none=True)
        loss=F.cross_entropy(model(x),y)
        loss.backward()
    else:
        loss=graph.replay(x,y)
    grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    if not torch.isfinite(grad_norm).item():
        raise RuntimeError('Non-finite gradients detected')
    opt.step()
    return loss

GRAPH_CHECKS=[]
def graph_equivalence_gate(arch,seed=303):
    # Compare one eager and graph step from identical initial parameters.
    eager=make_seeded_model(arch,seed)
    graphed=make_seeded_model(arch,seed)
    assert all(torch.equal(a,b) for a,b in zip(eager.parameters(),graphed.parameters()))
    x,y=parity_batch(BATCH,TRAIN_CONTEXT,314159,active_bits=16)
    oe=torch.optim.AdamW(eager.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    og=torch.optim.AdamW(graphed.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    capture=None
    try:
        capture=ForwardBackwardGraph(graphed,x,y)
        for run in (1,2):
            le=optimizer_step(eager,oe,x,y)
            lg=optimizer_step(graphed,og,x,y,capture)
            torch.cuda.synchronize()
            err=abs(float(le.item())-float(lg.item()))
            weight_err=max(float((a-b).abs().max().item()) for a,b in zip(eager.parameters(),graphed.parameters()))
            good=err<5e-4 and weight_err<5e-4
            GRAPH_CHECKS.append(dict(arch=arch,step=run,status='PASS' if good else 'FAIL',
                                    loss_error=err,weight_error=weight_err))
            if not good: raise AssertionError(f'{arch} CUDA Graph is not equivalent to eager at step {run}')
        return True
    finally:
        del capture,eager,graphed,oe,og
        gc.collect();torch.cuda.empty_cache()

GRAPH_AVAILABLE={}
for arch in ('slstm','pplstm'):
    if USE_CUDA_GRAPHS:
        try:
            GRAPH_AVAILABLE[arch]=graph_equivalence_gate(arch)
            print('CUDA Graph/eager training equivalence:',arch,'PASS')
        except Exception as e:
            GRAPH_AVAILABLE[arch]=False
            GRAPH_CHECKS.append(dict(arch=arch,step='capture',status='FAILED_FALLBACK_TO_EAGER',error=f'{type(e).__name__}: {str(e)[:400]}'))
            print('CUDA Graph unavailable for',arch,':',repr(e),'— eager fallback WILL BE LABELLED')
    else: GRAPH_AVAILABLE[arch]=False
save_csv(OUT/'graph_equivalence.csv',GRAPH_CHECKS)

## 8. Train each isolated block on the same parity task

In [ ]:
# Default runs are 2 architectures × 2 seeds, each with 8,000 optimizer steps.
# The curriculum (8→16→32→64→127 active bits) is identical between architectures.
# For the optional direct baseline, train with 127 active bits from step one.
# IMPORTANT: the padded short-bit phases still use 128 total token positions.

CURVES=[]
FINALS=[]
RUN_INFO=[]

def checkpoint_path(arch,seed,mode,root=OUT):
    return Path(root)/f'{arch}_{mode}_seed{seed}_latest.pt'

def train_one(arch,seed,mode='curriculum'):
    ident=f'{arch}-{mode}-seed{seed}'
    torch.manual_seed(seed);np.random.seed(seed);random.seed(seed)
    model=make_seeded_model(arch,seed).train()
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    start=0;elapsed_prior=0.0
    cp=checkpoint_path(arch,seed,mode,root=resume_root)
    if cp.exists():
        saved=torch.load(cp,map_location=DEVICE,weights_only=False)
        expected=dict(architecture=arch,seed=seed,mode=mode,width=WIDTH,
                      context=TRAIN_CONTEXT,batch=BATCH,steps=TRAIN_STEPS)
        for key,v in expected.items():
            assert saved.get(key)==v,f'Resume checkpoint mismatch: {key}'
        model.load_state_dict(saved['model'])
        opt.load_state_dict(saved['optimizer'])
        start=int(saved['step'])
        elapsed_prior=float(saved.get('train_seconds',0.))
        CURVES.extend(saved.get('curves',[]))
        print('Resuming',ident,'at step',start)
    x0,y0=parity_batch(BATCH,TRAIN_CONTEXT,seed*10_000+17,active_bits=8)
    graph=None;engine='eager'
    if GRAPH_AVAILABLE.get(arch):
        try:
            graph=ForwardBackwardGraph(model,x0,y0)
            engine='cuda_graph_forward_backward'
        except Exception as exc:
            print('Graph creation failed:',ident,repr(exc),'using explicitly labelled eager')
            graph=None
    if start>=TRAIN_STEPS:print(ident,'training already complete; refreshing final eval')
    begin=time.perf_counter()
    for step in range(start+1,TRAIN_STEPS+1):
        active=TRAIN_CONTEXT-1 if mode=='direct' else active_bits_for_step(step)
        # identical examples across architectures at each seed, mode, step
        x,y=parity_batch(BATCH,TRAIN_CONTEXT,seed*1_000_000+step,active_bits=active)
        loss=optimizer_step(model,opt,x,y,graph)
        if step%REPORT_EVERY==0 or step==TRAIN_STEPS:
            torch.cuda.synchronize()
            accuracy=evaluate(model,TRAIN_CONTEXT,INTERIM_EVAL_BATCHES)
            # Real 127 random bits for validation, not curriculum-padded bits.
            # Also report the current curriculum stage to see if short parity is learned.
            phase=accuracy if active==TRAIN_CONTEXT-1 else evaluate(
                model,TRAIN_CONTEXT,INTERIM_EVAL_BATCHES,active_bits=active)
            row=dict(architecture=arch,seed=seed,training_mode=mode,step=step,
                     active_training_bits=active,phase_accuracy_pct=phase['accuracy_pct'],
                     phase_cross_entropy=phase['eval_loss'],
                     training_loss=float(loss.item()),
                     elapsed_train_seconds=elapsed_prior+time.perf_counter()-begin,
                     engine=engine,params=count_params(model),**accuracy)
            CURVES[:] = [r for r in CURVES if not (r['architecture']==arch and r['seed']==seed and
                         r['training_mode']==mode and r['step']==step)]
            CURVES.append(row)
            save_csv(OUT/'learning_curves.csv',CURVES)
            print(f'{ident}: {step:,}/{TRAIN_STEPS:,} active={active:3} '
                  f'phase val={phase["accuracy_pct"]:.2f}% '
                  f'full-127-bit val={accuracy["accuracy_pct"]:.2f}% '
                  f'loss={loss.item():.3f} pred1={accuracy["pred_one_pct"]:.0f}% '
                  f'engine={engine}',flush=True)
            export_results()
        if step%SAVE_EVERY==0 or step==TRAIN_STEPS:
            torch.cuda.synchronize()
            # Save checkpoints outside the ZIP; optionally publish/download them separately.
            savefile=checkpoint_path(arch,seed,mode,root=OUT)
            atomic_save(dict(architecture=arch,seed=seed,mode=mode,
                    width=WIDTH,context=TRAIN_CONTEXT,batch=BATCH,steps=TRAIN_STEPS,
                    step=step,model=model.state_dict(),optimizer=opt.state_dict(),
                    train_seconds=elapsed_prior+time.perf_counter()-begin,
                    curves=[r for r in CURVES if r['architecture']==arch and r['seed']==seed and r['training_mode']==mode]),
                    savefile)
    torch.cuda.synchronize()
    elapsed=elapsed_prior+time.perf_counter()-begin
    RUN_INFO.append(dict(architecture=arch,seed=seed,training_mode=mode,
                         parameters=count_params(model),engine=engine,steps=TRAIN_STEPS,
                         training_seconds=elapsed))
    for length in EVAL_LENGTHS:
        try:
            score=evaluate(model,length,FINAL_EVAL_BATCHES)
            result=dict(architecture=arch,seed=seed,training_mode=mode,
                       train_context=TRAIN_CONTEXT,eval_context=length,steps=TRAIN_STEPS,
                       params=count_params(model),engine=engine,status='PASS',**score)
        except RuntimeError as ex:
            result=dict(architecture=arch,seed=seed,training_mode=mode,
                       eval_context=length,engine=engine,status='FAILED',error=str(ex)[:500])
            torch.cuda.empty_cache()
        FINALS.append(result)
        save_csv(OUT/'final_evaluations.csv',FINALS)
        print('FINAL',ident,length,result.get('accuracy_pct'),result['status'],flush=True)
        export_results()
    del model,opt,graph
    gc.collect();torch.cuda.empty_cache()

if SMOKE_ONLY:
    for arch in ('slstm','pplstm'):
        m=make_seeded_model(arch,10)
        x,y=parity_batch(BATCH,TRAIN_CONTEXT,19)
        o=torch.optim.AdamW(m.parameters(),lr=LR)
        value=optimizer_step(m,o,x,y)
        print('SMOKE',arch,'loss',float(value.item()))
        del m,o
    print('SMOKE_ONLY=True; full experiment not started.')
else:
    modes=('curriculum','direct') if RUN_DIRECT_BASELINES else ('curriculum',)
    for mode in modes:
        for seed in SEEDS:
            for arch in ('slstm','pplstm'):
                print('START',arch,'seed',seed,'mode',mode,flush=True)
                train_one(arch,seed,mode)

## 9. Summary, failure diagnostics and download

In [ ]:
CONFIG=dict(width=WIDTH,heads=HEADS,train_context=TRAIN_CONTEXT,batch=BATCH,
           train_steps=TRAIN_STEPS,seeds=SEEDS,lr=LR,weight_decay=WEIGHT_DECAY,
           evaluation_lengths=EVAL_LENGTHS,final_eval_batches=FINAL_EVAL_BATCHES,
           curriculum='right-aligned 8/16/32/64/127 active bits at fixed 128 token sequence',
           direct_baselines=RUN_DIRECT_BASELINES,slstm_backend='official xlstm vanilla',
           slstm_capture_patch='verified tensor-only global condition',
           pplstm_source='original PPLSTM AGPL-3.0-only cell',
           GPU=torch.cuda.get_device_name(0),torch_version=torch.__version__,
           optimized_slstm_cuda_kernel_used=False,
           cuda_graphs_requested=USE_CUDA_GRAPHS,
           graph_status=GRAPH_AVAILABLE,
           notes='One official residual xLSTM block each, embedding and final-token head; parameters NOT matched.')
save_json(OUT/'configuration.json',CONFIG)
save_csv(OUT/'scan_checks.csv',SCAN_CHECKS)
save_csv(OUT/'slstm_patch_equivalence.csv',PATCH_CHECKS)
save_csv(OUT/'graph_equivalence.csv',GRAPH_CHECKS)
save_csv(OUT/'run_metadata.csv',RUN_INFO)
save_csv(OUT/'learning_curves.csv',CURVES)
save_csv(OUT/'final_evaluations.csv',FINALS)
archive=export_results()
if FINALS:
    final_df=pd.DataFrame(FINALS)
    display(final_df[['architecture','seed','training_mode','eval_context','accuracy_pct',
                      'pred_one_pct','true_zero_recall_pct','true_one_recall_pct','engine']].sort_values(['training_mode','seed','eval_context','architecture']))
    print('Mean final accuracy by model and length (multiple seeds are descriptive, not a confidence interval):')
    display(final_df.groupby(['training_mode','architecture','eval_context'],as_index=False)['accuracy_pct'].agg(['mean','std']).reset_index())
else:
    print('No full training evaluations (SMOKE_ONLY or interrupted run).')
print('Results archive:',archive,'bytes:',archive.stat().st_size)
print('Checkpoint files (NOT in ZIP):',list(OUT.glob('*_latest.pt')))
print('Download from Kaggle /kaggle/working/single_block_parity/ or Kaggle output tab.')

### How to read your results

1. Check `slstm_patch_equivalence.csv`, `scan_checks.csv` and `graph_equivalence.csv`: numerical checks must pass before trusting speed comparisons.
2. The learning curve shows both `phase_accuracy_pct` (current active-bit difficulty) and `accuracy_pct` (full 127 random bits). Compare **held-out parity at 128 tokens** after the full-bit phase. Near 50% with ~50/50 class predictions means the task is unsolved; near 50% with **100% of predictions in one class** shows class collapse. `true_zero_recall_pct` and `true_one_recall_pct` make this visible.
3. Compare 256 and 512 to test **generalization beyond the 128-token training length**. The curriculum uses left-zero-padded *active bits*, so these tests are harder in both sequence length and number of random input bits.
4. Compare across seeds before calling one model better. The sLSTM and PPLSTM blocks have **different parameter counts**. No mLSTM is included, so this isolates the one-block choices much more closely than the earlier combined model.
5. `PPLSTM_sLSTM_Single_Block_Parity_Results.zip` contains JSON/CSV results, not large weight checkpoints. Kaggle keeps `.pt` files in the output folder, and resume requires attaching saved files to a new session and pointing `RESUME_DIR` to them.

**Not tested:** optimized official sLSTM CUDA kernels (these commonly require compute capability ≥8.0; a Kaggle T4 is 7.5). CUDA Graphs improve launch overhead; they do not make your parallel scan a custom fused CUDA kernel.

**If parity remains at chance:** the next controlled check is a bit-by-bit parity readout (supervision at every time step rather than only the final query); this is a different training task and should be reported separately.